# Task 3 · Generate the team's GSO dataset
Run on a CPU runtime. Mount your project's Google Drive in the first cell. The second cell times the first archive. The third processes the remaining archives. Completed candidate archives are saved to Drive for resuming. The fourth cell uses Marko's ranking, split and corruption code and checks the finished dataset before saving it to Drive.


In [ ]:
from pathlib import Path
import io, json, os, shutil, sys, tarfile, time, zipfile
from collections import Counter
from types import SimpleNamespace
from importlib.metadata import version
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'trimesh==5.2.2'], check=True)
import numpy as np
import requests
import trimesh
from google.colab import drive

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/MVP_Project/task3')
REV = 'b944bf0e56d93c82652b8b7243187be237434ed4'
POINTS = 10000
SEED = 2025
root = Path('/content/team_gso')
root.mkdir(exist_ok=True)
cache = DRIVE / f'candidates_{REV[:8]}_{POINTS}_{SEED}_v1'
cache.mkdir(parents=True, exist_ok=True)
target = DRIVE / 'gso_task3_v1'

scripts = {'loading': 'import io\nimport trimesh\nimport hashlib\nimport numpy as np\nfrom trimesh.visual.color import uv_to_interpolated_color\n\nfrom corruption import add_colour_noise, add_geometry_noise\nfrom visualization import show_mesh, show_cloud\n\nDATASET = "suvadityamuk/google-scanned-objects"\n\ndef stable_seed(seed, object_id, label):\n    value = f"{seed}:{object_id}:{label}".encode()\n    digest = hashlib.sha256(value).digest()\n    return int.from_bytes(digest[:8], "little")\n\ndef load_glb(asset):\n    if isinstance(asset, (trimesh.Trimesh, trimesh.Scene)):\n        mesh = asset\n    elif isinstance(asset, dict):\n        if asset.get("bytes") is not None:\n            mesh = trimesh.load(io.BytesIO(asset["bytes"]), file_type="glb", force="scene")\n        else:\n            mesh = trimesh.load(asset["path"], force="scene")\n    else:\n        mesh = trimesh.load(io.BytesIO(asset), file_type="glb", force="scene")\n\n    # GLB may contain several separate mesh components.\n    if isinstance(mesh, trimesh.Scene):\n        meshes = [g for g in mesh.dump() if isinstance(g, trimesh.Trimesh)]\n    else:\n        meshes = [mesh]\n\n    return meshes\n\n\ndef sample_coloured_cloud(meshes, count, rng):\n    areas = np.array([m.area for m in meshes])\n    counts = rng.multinomial(count, areas / areas.sum())\n\n    points_list = []\n    colors_list = []\n\n    for geom, count in zip(meshes, counts):\n        if count == 0:\n            continue\n\n        # Sample XYZ positions and corresponding barycentric coordinates.\n        points, face_indices, bary = trimesh.sample.sample_surface(\n            geom, count, return_barycentric=True, seed=int(rng.integers(0, 2**31)),\n        )\n\n        visual = geom.visual\n\n        if isinstance(visual, trimesh.visual.TextureVisuals):\n            # Interpolate the UV coordinates of each sampled point.\n            uv_vertices = visual.uv[geom.faces[face_indices]]\n            uv = np.einsum("nij,ni->nj", uv_vertices, bary)\n\n            material = visual.material\n\n            # Support both traditional and GLB/PBR texture materials.\n            texture = getattr(material, "image", None)\n            if texture is None:\n                texture = getattr(material, "baseColorTexture", None)\n\n            if texture is None:\n                raise ValueError("No colour texture found for mesh component")\n\n            colors = uv_to_interpolated_color(uv, texture)[:, :3]\n\n        elif visual.kind == "vertex":\n            vertex_colors = visual.vertex_colors[geom.faces[face_indices], :3]\n            colors = np.einsum("nij,ni->nj", vertex_colors.astype(float), bary)\n\n        elif visual.kind == "face":\n            colors = visual.face_colors[face_indices, :3]\n\n        else:\n            raise ValueError("Mesh component has no usable colour data")\n\n        points_list.append(points)\n        colors_list.append(colors)    \n\n    points = np.concatenate(points_list)\n    colors = np.concatenate(colors_list).astype(np.float32) / 255.0\n\n    return points, colors\n\n', 'visualization': '\nimport matplotlib\nmatplotlib.use("Agg")\n\nimport matplotlib.pyplot as plt\nimport numpy as np\n\ndef show_mesh(mesh):\n    """Interactively display a textured mesh."""\n    mesh.show()\n\n\ndef show_cloud(xyz, rgb, title="Point cloud"):\n    """Interactively display XYZRGB points using Open3D."""\n    import open3d as o3d\n\n    pcd = o3d.geometry.PointCloud()\n    pcd.points = o3d.utility.Vector3dVector(xyz)\n    pcd.colors = o3d.utility.Vector3dVector(rgb)\n\n    o3d.visualization.draw_geometries([pcd], window_name=title)\n\n\ndef save_comparison(\n    path, clean_xyz, noisy_xyz, clean_rgb, noisy_rgb\n):\n    """Save side-by-side clean and noisy point-cloud previews."""\n    fig = plt.figure(figsize=(10, 5))\n\n    clouds = [\n        (clean_xyz, clean_rgb, "Clean XYZRGB"),\n        (noisy_xyz, noisy_rgb, "Noisy XYZRGB"),\n    ]\n\n    for i, (xyz, rgb, title) in enumerate(clouds):\n        ax = fig.add_subplot(1, 2, i + 1, projection="3d")\n\n        ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], c=rgb, s=0.2, rasterized=True)\n        ax.set_title(title)\n        ax.set_box_aspect(np.maximum(np.ptp(clean_xyz, axis=0), 1e-6))\n        ax.set_axis_off()\n        ax.view_init(elev=20, azim=45)\n\n    fig.tight_layout()\n    fig.savefig(path, dpi=130)\n    plt.close(fig)', 'ranking': 'import numpy as np\nimport pandas as pd\nfrom sklearn.neighbors import NearestNeighbors\n\nDATASET_COMPOSITION = (20, 10, 10, 20) # agreement, disagreement (geometry varies), disagreement (texture varies), random samples\n\ndef compute_scores(xyz, rgb, k=24):\n    # Find neighbours in clean 3D geometry.\n    knn = NearestNeighbors(n_neighbors=k + 1, algorithm="kd_tree").fit(xyz)\n    indices = knn.kneighbors(xyz, return_distance=False)[:, 1:]\n    neighbours = xyz[indices]  # (N, k, 3)\n\n    # Local geometric variation via PCA.\n    centered = neighbours - neighbours.mean(axis=1, keepdims=True)\n    covariance = (np.einsum("nki,nkj->nij", centered, centered) / k)\n    eigenvalues = np.linalg.eigvalsh(covariance)\n    geometry_score = (eigenvalues[:, 0] / (eigenvalues.sum(axis=1) + 1e-12))\n\n    # Local RGB variation.\n    color_distances = np.linalg.norm(rgb[indices] - rgb[:, None, :], axis=2)\n    color_score = color_distances.mean(axis=1)\n\n    return geometry_score, color_score\n\ndef compute_thresholds(objects):\n    """\n    objects: dict mapping object_id -> (geometry_scores, colour_scores)\n    """\n    all_g = np.concatenate([g for g, c in objects.values()])\n    all_c = np.concatenate([c for g, c in objects.values()])\n\n    return {\n        "g_high": np.quantile(all_g, 0.90),\n        "g_low": np.quantile(all_g, 0.50),\n        "c_high": np.quantile(all_c, 0.90),\n        "c_low": np.quantile(all_c, 0.50),\n    }\n\n\ndef summarize_object(g, c, thresholds):\n    gh = thresholds["g_high"]\n    gl = thresholds["g_low"]\n    ch = thresholds["c_high"]\n    cl = thresholds["c_low"]\n\n    texture_only = np.mean((c >= ch) & (g <= gl))\n    geometry_only = np.mean((g >= gh) & (c <= cl))\n    aligned = np.mean((g >= gh) & (c >= ch))\n\n    return {\n        "texture_only": float(texture_only),\n        "geometry_only": float(geometry_only),\n        "aligned": float(aligned),\n        "disagreement": float(texture_only + geometry_only),\n    }\n\ndef select_objects(ranking, seed=2025, composition=DATASET_COMPOSITION):\n    selected = set()\n    groups = {}\n\n    agreement, geometry_only, texture_only, random_samples = composition\n\n    criteria = [\n        ("agreement", "aligned", agreement),\n        ("texture_only", "texture_only", texture_only),\n        ("geometry_only", "geometry_only", geometry_only),\n    ]\n\n    for group, column, count in criteria:\n        ranked = ranking.sort_values([column, "object_id"], ascending=[False, True])\n        choices = [obj for obj in ranked["object_id"] if obj not in selected][:count]\n\n        if len(choices) != count:\n            raise ValueError(f"Not enough objects for {group}")\n\n        groups[group] = choices\n        selected.update(choices)\n\n    remaining = ranking.loc[~ranking["object_id"].isin(selected), "object_id", ].to_numpy()\n    rng = np.random.default_rng(seed)\n    groups["random"] = list(rng.choice(remaining, size=random_samples, replace=False))\n\n    return groups\n\ndef create_splits(groups, seed=2025):\n    rng = np.random.default_rng(seed)\n    assignments = {}\n\n    for group, ids in groups.items():\n        ids = list(ids)\n        rng.shuffle(ids)\n\n        n_train = int(len(ids) * 0.7)\n        n_val = int(len(ids) * 0.1)\n\n        for i, object_id in enumerate(ids):\n            split = (\n                "train" if i < n_train\n                else "val" if i < n_train + n_val\n                else "test"\n            )\n\n            assignments[object_id] = {"split": split, "group": group}\n\n    return assignments\n\ndef rank_candidates(candidates):\n    objects = {}\n\n    for candidate in candidates:\n        points, colors, meta = candidate\n        object_id = str(meta["object_id"])\n        objects[object_id] = compute_scores(points, colors)\n\n    thresholds = compute_thresholds(objects)\n\n    rows = [\n            {"object_id": object_id,\n             **summarize_object(g, c, thresholds)}\n            for object_id, (g, c) in objects.items()\n        ]\n    \n    ranking = pd.DataFrame(rows)\n    return ranking', 'corruption': 'import numpy as np\nfrom pathlib import Path\n\ndef bounding_radius(xyz):\n    center = (xyz.max(axis=0) + xyz.min(axis=0)) / 2\n    return np.linalg.norm(xyz - center, axis=1).max()\n\n\ndef add_geometry_noise(xyz, level, rng):\n    sigma = level * bounding_radius(xyz)\n    return (xyz + rng.normal(0, sigma, size=xyz.shape)).astype(np.float32)\n\n\ndef add_colour_noise(rgb, level, rng):\n    return np.clip(rgb + rng.normal(0, level, size=rgb.shape), 0, 1).astype(np.float32)\n\n\n# Running the script generates 2.5% noise point clouds that were missing from data.zip\n', 'generate': 'import argparse\nfrom pathlib import Path\nimport numpy as np\nimport json\n\nfrom loading import stable_seed\nfrom corruption import add_geometry_noise, add_colour_noise\nfrom visualization import save_comparison\nfrom ranking import select_objects, create_splits, rank_candidates\n\n\ndef parse_levels(value):\n    return [float(x) for x in value.split(",")]\n\ndef generate_object(candidate, info, root, args):\n    xyz, rgb, meta = candidate\n    object_id = str(meta["object_id"])\n\n    folder = root / info["split"] / object_id\n    folder.mkdir(parents=True, exist_ok=True)\n\n    np.savez_compressed(folder / "clean.npz", xyz=xyz, rgb=rgb)\n\n    for xyz_level in args.xyz_noise:\n        for rgb_level in args.rgb_noise:\n            label = f"noise_xyz_{xyz_level:g}_rgb_{rgb_level:g}"\n\n            rng_xyz = np.random.default_rng(stable_seed(args.seed, object_id, f"xyz_{xyz_level:g}"))\n            rng_rgb = np.random.default_rng(stable_seed(args.seed, object_id, f"rgb_{rgb_level:g}"))\n\n            noisy_xyz = add_geometry_noise(xyz, xyz_level, rng_xyz)\n            noisy_rgb = add_colour_noise(rgb, rgb_level, rng_rgb)\n\n            np.savez_compressed(folder / f"{label}.npz", xyz=noisy_xyz, rgb=noisy_rgb)\n\n            if args.preview:\n                save_comparison(folder / f"{label}.png", xyz, noisy_xyz, rgb, noisy_rgb)\n\ndef generate_objects(assignments, candidates, args):\n    root = Path(args.output)\n    root.mkdir(parents=True, exist_ok=True)\n\n    for candidate in candidates:\n        object_id = str(candidate[2]["object_id"])\n\n        if object_id in assignments:\n            generate_object(candidate, assignments[object_id], root, args)\n\ndef save_selection(assignments, args):\n    selection_path = Path(args.output) / "selection.json"\n    selection_path.parent.mkdir(parents=True, exist_ok=True)\n\n    if selection_path.exists():\n        existing = json.loads(selection_path.read_text())\n        if existing != assignments:\n            raise RuntimeError(\n                "Existing dataset uses different object assignments. "\n                "Choose another --output directory."\n            )\n    else:\n        selection_path.write_text(json.dumps(assignments, indent=2))\n\n'}

for name, source in scripts.items():
    (root / f'{name}.py').write_text(source)
sys.path.insert(0, str(root))
from loading import load_glb, sample_coloured_cloud, stable_seed
from ranking import rank_candidates, select_objects, create_splits
from generate import generate_objects, save_selection

def save_candidate_archive(index):
    filename = f'gso-train-{index:05d}.tar'
    dest = cache / f'{index:05d}.npz'
    if dest.is_file():
        with np.load(dest) as saved:
            return len(saved['ids']), len(saved['skipped']), 0
    started = time.monotonic()
    archive = root / filename
    url = f'https://huggingface.co/datasets/hugging-3d/google-scanned-objects/resolve/{REV}/data/{filename}'
    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=(20, 90)) as reply:
                reply.raise_for_status()
                with archive.open('wb') as stream:
                    for chunk in reply.iter_content(1024 * 1024):
                        stream.write(chunk)
            break
        except (requests.RequestException, OSError):
            if attempt == 2:
                raise
            time.sleep(3 * (attempt + 1))
    ids, points, colours, skipped = [], [], [], []
    with tarfile.open(archive) as tar:
        members = {}
        for entry in tar.getmembers():
            if entry.isfile():
                key, _, ext = Path(entry.name).name.partition('.')
                if ext in ('json', 'glb'):
                    members.setdefault(key, {})[ext] = entry
        if not members:
            raise ValueError(f'{filename} contains no objects')
        for key, pair in members.items():
            if set(pair) != {'glb', 'json'}:
                skipped.append(f'{key}: missing GLB or metadata')
                continue
            meta = json.load(tar.extractfile(pair['json']))
            oid = str(meta['object_id'])
            try:
                glb = tar.extractfile(pair['glb']).read()
                rng = np.random.default_rng(stable_seed(SEED, oid, 'surface'))
                xyz, rgb = sample_coloured_cloud(load_glb(glb), POINTS, rng)
                if (xyz.shape != (POINTS, 3) or rgb.shape != xyz.shape or
                    not np.isfinite(xyz).all() or not np.isfinite(rgb).all() or
                    rgb.min() < 0 or rgb.max() > 1 or np.ptp(xyz, axis=0).max() <= 0):
                    raise ValueError('Invalid sampled cloud')
            except (ValueError, TypeError, AttributeError, IndexError, KeyError) as exc:
                skipped.append(f'{oid}: {exc}')
                continue
            ids.append(oid)
            points.append(xyz)
            colours.append(rgb)
    if not ids:
        raise RuntimeError(f'{filename}: no valid meshes. First errors: {skipped[:3]}')
    local = root / 'candidates.npz'
    np.savez_compressed(local, ids=ids, xyz=np.stack(points), rgb=np.stack(colours), skipped=np.asarray(skipped, dtype=str))
    part = dest.with_suffix('.part')
    shutil.copyfile(local, part)
    part.replace(dest)
    archive.unlink()
    return len(ids), len(skipped), time.monotonic() - started


In [ ]:
count, skipped, seconds = save_candidate_archive(0)
print(f'Archive 1/43 · {count} sampled · {skipped} skipped · {seconds:.0f} s', flush=True)
if seconds:
    print(f'Rough processing estimate from the first archive: {seconds * 43 / 60:.0f} min, plus ranking and Drive export.')


In [ ]:
for index in range(1, 43):
    count, skipped, seconds = save_candidate_archive(index)
    state = f'{seconds:.0f} s' if seconds else 'reused'
    print(f'Archive {index+1}/43 · {count} sampled · {skipped} skipped · {state}', flush=True)


In [ ]:
candidates, failures = [], []
for index in range(43):
    with np.load(cache / f'{index:05d}.npz') as f:
        candidates.extend((xyz, rgb, {'object_id': str(oid)}) for oid, xyz, rgb in zip(f['ids'], f['xyz'], f['rgb']))
        failures.extend(f['skipped'].tolist())
ids = [meta['object_id'] for _, _, meta in candidates]
assert len(set(ids)) == len(ids), 'Duplicate object IDs'
assert len(ids) + len(failures) == 1030, 'Source object count differs from the pinned dataset'
print(f'Ranking {len(ids)} candidate objects. {len(failures)} could not be sampled.', flush=True)
ranking = rank_candidates(candidates)
groups = select_objects(ranking, seed=SEED)
assignments = create_splits(groups, seed=SEED)
assert len(assignments) == 60 and sum(map(len, groups.values())) == 60
staged = root / 'dataset'
args = SimpleNamespace(output=str(staged), seed=SEED, xyz_noise=[0.01,0.02,0.025], rgb_noise=[0.,0.05], preview=False)
save_selection(assignments, args)
generate_objects(assignments, candidates, args)

def verify(folder):
    splits = Counter()
    for oid, info in assignments.items():
        split = info['split']
        splits[split] += 1
        path = folder / split / oid
        xyz_for_level = {}
        rgb_for_level = {}
        configs = [(x, r) for x in args.xyz_noise for r in args.rgb_noise]
        files = ['clean.npz'] + [f'noise_xyz_{x:g}_rgb_{r:g}.npz' for x, r in configs]
        for name in files:
            with np.load(path / name) as f:
                assert f['xyz'].shape == f['rgb'].shape == (POINTS, 3)
                assert np.isfinite(f['xyz']).all() and np.isfinite(f['rgb']).all()
                assert np.min(f['rgb']) >= 0 and np.max(f['rgb']) <= 1
        for x, r in configs:
            with np.load(path / f'noise_xyz_{x:g}_rgb_{r:g}.npz') as f:
                assert np.array_equal(f['xyz'], xyz_for_level.setdefault(x, f['xyz']))
                assert np.array_equal(f['rgb'], rgb_for_level.setdefault(r, f['rgb']))
    assert splits == {'train': 42, 'val': 6, 'test': 12}
    return splits

splits = verify(staged)
manifest = dict(source='hugging-3d/google-scanned-objects', revision=REV, seed=SEED,
                points=POINTS, xyz_noise=args.xyz_noise, rgb_noise=args.rgb_noise,
                splits=dict(splits), groups={k:len(v) for k,v in groups.items()},
                skipped=failures, dependencies={p:version(p) for p in ('trimesh','numpy','scikit-learn')})
(staged/'manifest.json').write_text(json.dumps(manifest, indent=2))
if target.exists():
    assert json.loads((target/'selection.json').read_text()) == assignments, 'Existing output uses a different selection'
    assert json.loads((target/'manifest.json').read_text()) == manifest, 'Existing output uses a different configuration'
else:
    part = DRIVE / '.gso_task3_v1_pending'
    shutil.copytree(staged, part, dirs_exist_ok=True)
    verify(part)
    part.rename(target)
verify(target)
print(f'COMPLETE · {len(assignments)} objects · {dict(splits)} · {target}')
